# FloodWatch – Mapping flood damage from space (Multimodal AI Hackathon 2026, Track B)
Educational prototype, not an operational tool.

**Attribution (required):**
Contains modified Copernicus Sentinel data 2026.
Produced using Copernicus WorldDEM-30 © DLR e.V. 2010–2014 and © Airbus Defence and Space GmbH 2014–2018 provided under COPERNICUS by the European Union and ESA; all rights reserved.
© OpenStreetMap contributors. (ODbL)
If you use Kuro Siwo / Sen1Floods11 for training, cite Bountos et al., 2024 / Bonafilia et al., CVPR Workshops 2020.

**Pipeline:** Sentinel-1 (same-track before/after) + Sentinel-2 -> flood & debris mask -> OSM damage
-> road graph cut-off analysis -> DEM flood-path trace (bonus) -> LLM situation-report copilot (EN/NE)
-> folium dashboard + one-page HTML report.

## 1. Install

In [30]:
!pip -q install rasterio geopandas shapely pyproj networkx scipy folium anthropic requests matplotlib

## 2. Configuration & secrets
Add these in Colab -> left sidebar key icon ("Secrets"):
Keys are filled in the next cell (Copernicus OAuth client + Groq key).

In [31]:
import os, io, re, json, math, time, getpass, base64, datetime as dt
import numpy as np
import requests
import rasterio
from rasterio import features
from rasterio.io import MemoryFile
from scipy import ndimage as ndi
from scipy.spatial import cKDTree
import geopandas as gpd
import shapely
from shapely.geometry import shape, LineString
import networkx as nx
from pyproj import Transformer
import folium
import matplotlib
import matplotlib.pyplot as plt

CONFIG = dict(
    # Area of interest (west, south, east, north) in lon/lat. Example: Bhote Koshi-Trishuli corridor (approximate, adjust it!)
    bbox=(85.10, 27.90, 85.45, 28.32),
    flood_date="2026-08-26",          # day of the event
    osm_date="2026-07-27",            # MUST be before the flood date (ohsome snapshot)
    res=30,                           # metres per pixel (auto-coarsened if the area is too big)
    upstream_point=(85.38, 28.28),    # (lon, lat) for the flood-path bonus; set None to skip
    out_dir="outputs",
    # thresholds
    water_vv_db=-18.0, water_vh_db=-24.0, change_db=3.0,
    max_slope_deg=25.0, max_hand_m=60.0, min_blob_px=8,
    use_optical=True,
)
os.makedirs(CONFIG["out_dir"], exist_ok=True)
assert CONFIG["osm_date"] < CONFIG["flood_date"], "OSM snapshot must be BEFORE the event (data rules)."

def get_secret(name, required=True):
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v: return v
    except Exception:
        pass
    v = os.environ.get(name)
    if not v and required:
        v = getpass.getpass(f"{name}: ")
    return v

# Keys pasted directly (personal copy only - do NOT upload this notebook to GitHub with the keys inside)
CDSE_ID = "sh-72ea293d-97b7-4d06-952f-e264dd31f680"
CDSE_SECRET = "DzTfCSEcDKfb3HBlYEKkL2Be1qh7EuxlPKE6gZlE7cE9ige2rFjRl8nPhZxME9jYT8jHmfbMlQocnW07k71Thc"
GROQ_KEY = "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"
ANTHROPIC_KEY = GROQ_KEY   # name kept so ask() treats the copilot as enabled

## 3. Copernicus Data Space (Sentinel Hub Process API) helpers
One API gives us orthorectified, terrain-corrected Sentinel-1, Sentinel-2 L2A and Copernicus DEM,
already cropped and resampled to a common grid.

In [32]:
TOKEN_URL = "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token"
PROCESS_URL = "https://sh.dataspace.copernicus.eu/api/v1/process"
CATALOG_URL = "https://sh.dataspace.copernicus.eu/api/v1/catalog/1.0.0/search"
OHSOME_URL = "https://api.ohsome.org/v1/elements/geometry"

_tok = {"v": None, "t": 0.0}
def auth_headers():
    if not _tok["v"] or time.time() - _tok["t"] > 420:
        r = requests.post(TOKEN_URL, data=dict(grant_type="client_credentials",
                          client_id=CDSE_ID, client_secret=CDSE_SECRET), timeout=60)
        r.raise_for_status()
        _tok.update(v=r.json()["access_token"], t=time.time())
    return {"Authorization": f"Bearer {_tok['v']}"}

class Grid:
    """Common metric (UTM) grid used by every raster."""
    def __init__(self, bbox, res):
        w, s, e, n = bbox
        lon0, lat0 = (w + e) / 2, (s + n) / 2
        self.epsg = (32600 if lat0 >= 0 else 32700) + int((lon0 + 180) // 6) + 1
        tr = Transformer.from_crs(4326, self.epsg, always_xy=True)
        pts = [tr.transform(x, y) for x, y in [(w, s), (w, n), (e, s), (e, n)]]
        minx, maxx = min(p[0] for p in pts), max(p[0] for p in pts)
        miny, maxy = min(p[1] for p in pts), max(p[1] for p in pts)
        while max((maxx - minx), (maxy - miny)) / res > 2400:
            res *= 1.25                                   # Process API limit is 2500 px
        self.res = res
        self.W = int(math.ceil((maxx - minx) / res)); self.H = int(math.ceil((maxy - miny) / res))
        self.bounds = (minx, miny, minx + self.W * res, miny + self.H * res)
        self.transform = rasterio.transform.from_origin(minx, miny + self.H * res, res, res)
        self.to_ll = Transformer.from_crs(self.epsg, 4326, always_xy=True)
        self.from_ll = tr

def process(data, evalscript, grid):
    body = {"input": {"bounds": {"bbox": list(grid.bounds),
                                 "properties": {"crs": f"http://www.opengis.net/def/crs/EPSG/0/{grid.epsg}"}},
                      "data": data},
            "output": {"width": grid.W, "height": grid.H,
                       "responses": [{"identifier": "default", "format": {"type": "image/tiff"}}]},
            "evalscript": evalscript}
    r = requests.post(PROCESS_URL, json=body, headers=auth_headers(), timeout=600)
    if r.status_code != 200:
        raise RuntimeError(f"Process API {r.status_code}: {r.text[:400]}")
    with MemoryFile(r.content) as mf, mf.open() as ds:
        return ds.read().astype("float32")

EVAL_DEM = """//VERSION=3
function setup(){return{input:["DEM"],output:{bands:1,sampleType:"FLOAT32"}}}
function evaluatePixel(s){return[s.DEM]}"""
EVAL_S1 = """//VERSION=3
function setup(){return{input:["VV","VH","dataMask"],output:{bands:3,sampleType:"FLOAT32"}}}
function evaluatePixel(s){return[s.VV,s.VH,s.dataMask]}"""
EVAL_S2 = """//VERSION=3
function setup(){return{input:["B03","B04","B08","B11","SCL","dataMask"],output:{bands:6,sampleType:"FLOAT32"}}}
function evaluatePixel(s){return[s.B03,s.B04,s.B08,s.B11,s.SCL,s.dataMask]}"""

def get_dem(grid):
    d = [{"type": "dem", "dataFilter": {"demInstance": "COPERNICUS_30"}}]
    dem = process(d, EVAL_DEM, grid)[0]
    dem[~np.isfinite(dem)] = np.nanmedian(dem)
    return dem

def s1_scene(grid, day, orbit):
    flt = {"timeRange": {"from": f"{day}T00:00:00Z", "to": f"{day}T23:59:59Z"},
           "acquisitionMode": "IW", "polarization": "DV", "resolution": "HIGH"}
    if orbit: flt["orbitDirection"] = orbit
    d = [{"type": "sentinel-1-grd", "dataFilter": flt,
          "processing": {"orthorectify": True, "demInstance": "COPERNICUS_30",
                         "backCoeff": "GAMMA0_TERRAIN", "speckleFilter": {"type": "NONE"}}}]
    return process(d, EVAL_S1, grid)

def s2_composite(grid, t0, t1):
    d = [{"type": "sentinel-2-l2a", "dataFilter": {
          "timeRange": {"from": f"{t0}T00:00:00Z", "to": f"{t1}T23:59:59Z"},
          "maxCloudCoverage": 80, "mosaickingOrder": "leastCC"}}]
    return process(d, EVAL_S2, grid)

def find_s1_pair(bbox, flood_day):
    """Find a before/after pair on the SAME orbit track (same direction, same time of day, multiple of 12 days apart)."""
    fd = dt.date.fromisoformat(flood_day)
    body = {"collections": ["sentinel-1-grd"], "bbox": list(bbox), "limit": 100,
            "datetime": f"{fd - dt.timedelta(days=48)}T00:00:00Z/{fd + dt.timedelta(days=24)}T23:59:59Z"}
    feats = []
    while True:
        r = requests.post(CATALOG_URL, json=body, headers=auth_headers(), timeout=120)
        r.raise_for_status(); j = r.json(); feats += j.get("features", [])
        nxt = j.get("context", {}).get("next")
        if not nxt: break
        body["next"] = nxt
    rows = []
    for f in feats:
        p = f["properties"]
        t = dt.datetime.fromisoformat(p["datetime"].replace("Z", "+00:00"))
        if p.get("sar:instrument_mode", "IW") != "IW": continue
        rows.append(dict(day=t.date(), tod=t.hour + t.minute / 60,
                         orbit=(p.get("sat:orbit_state") or "").upper() or None,
                         rel=p.get("sat:relative_orbit")))
    best = None
    for p in rows:
        if p["day"] <= fd: continue
        for q in rows:
            if q["day"] >= fd: continue
            dd = (p["day"] - q["day"]).days
            if (q["orbit"] == p["orbit"] and abs(q["tod"] - p["tod"]) < 0.3 and dd % 12 == 0
                    and (q["rel"] is None or p["rel"] is None or q["rel"] == p["rel"])):
                key = ((p["day"] - fd).days, (fd - q["day"]).days)
                if best is None or key < best[0]:
                    best = (key, q, p)
    if best is None:
        avail = sorted({(r["day"].isoformat(), r["orbit"], round(r["tod"], 1)) for r in rows})
        raise RuntimeError(f"No same-track S1 pair found. Available scenes (day, orbit, hour): {avail}")
    _, q, p = best
    return dict(pre_day=q["day"].isoformat(), post_day=p["day"].isoformat(), orbit=p["orbit"], rel=p["rel"])

## 4. Q1 – Where did the flood hit? (Sentinel-1 change detection + Sentinel-2 optical + terrain constraints)

In [33]:
def terrain(dem, res):
    gy, gx = np.gradient(ndi.gaussian_filter(dem, 1), res)
    return np.degrees(np.arctan(np.hypot(gx, gy)))

def hand_proxy(dem, seeds):
    """Height above the nearest permanent-water pixel (rough HAND)."""
    if seeds.sum() < 20:
        seeds = dem <= np.percentile(dem, 2)
    idx = ndi.distance_transform_edt(~seeds, return_distances=False, return_indices=True)
    return dem - dem[idx[0], idx[1]]

def s1_db(arr):
    """3-band (VV, VH, mask) -> speckle-reduced dB (NaN where invalid). Filter is done in linear power."""
    out = []
    valid = (arr[2] > 0) & (arr[0] > 0)
    for b in (arr[0], arr[1]):
        x = np.where(valid, b, 0.0)
        num = ndi.uniform_filter(x, 5); den = ndi.uniform_filter(valid.astype("float32"), 5)
        lin = np.where(den > 0.3, num / np.maximum(den, 1e-6), np.nan)
        out.append(10 * np.log10(np.clip(lin, 1e-5, None)))
    return out[0], out[1]

def clean(mask, min_px):
    mask = ndi.binary_opening(mask, structure=np.ones((3, 3)))
    lab, n = ndi.label(mask)
    if n == 0: return mask
    size = np.bincount(lab.ravel()); keep = size >= min_px; keep[0] = False
    return ndi.binary_fill_holes(keep[lab])

def detect_s1(pre, post, slope, hand, C):
    vv0, vh0 = s1_db(pre); vv1, vh1 = s1_db(post)
    ok = np.isfinite(vv0) & np.isfinite(vv1)
    perm_water = (vv0 < C["water_vv_db"]) & (slope < 5)
    water_pre = vv0 < C["water_vv_db"]
    shadow = vv0 < -22                                   # radar shadow / very dark before -> ignore
    zone = (slope < C["max_slope_deg"]) & (hand < C["max_hand_m"]) & ok
    new_water = (vv1 < C["water_vv_db"]) & (vh1 < C["water_vh_db"]) & ~water_pre & zone
    d = vv1 - vv0
    debris = (np.abs(d) > C["change_db"]) & ~new_water & ~shadow & zone
    return clean(new_water, C["min_blob_px"]), clean(debris, C["min_blob_px"]), perm_water, ok

def detect_s2(pre, post, slope, hand, C):
    def ndi_(a, b): return (a - b) / (a + b + 1e-6)
    def clear(x): return (x[5] > 0) & ~np.isin(x[4], [0, 1, 3, 8, 9, 10, 11])
    valid = clear(pre) & clear(post)
    frac = float(valid.mean())
    zone = valid & (slope < C["max_slope_deg"]) & (hand < C["max_hand_m"])
    mndwi0, mndwi1 = ndi_(pre[0], pre[3]), ndi_(post[0], post[3])
    ndvi0, ndvi1 = ndi_(pre[2], pre[1]), ndi_(post[2], post[1])
    water = (mndwi1 > 0.1) & (mndwi0 < 0.0) & zone
    debris = ((ndvi0 - ndvi1) > 0.3) & (ndvi1 < 0.25) & ~water & zone
    return clean(water, C["min_blob_px"]), clean(debris, C["min_blob_px"]), frac

def map_flood(grid, C):
    dem = get_dem(grid); slope = terrain(dem, grid.res)
    pair = find_s1_pair(C["bbox"], C["flood_date"])
    print("S1 pair:", pair)
    pre = s1_scene(grid, pair["pre_day"], pair["orbit"]); post = s1_scene(grid, pair["post_day"], pair["orbit"])
    vv0, _ = s1_db(pre)
    seeds = (vv0 < C["water_vv_db"]) & (slope < 5)
    seeds = ndi.binary_opening(seeds, np.ones((3, 3)))
    hand = hand_proxy(dem, seeds)
    w1, d1, perm, ok = detect_s1(pre, post, slope, hand, C)
    info = dict(s1_pre=pair["pre_day"], s1_post=pair["post_day"], orbit=pair["orbit"],
                s1_valid_pct=round(100 * float(ok.mean()), 1), s2_used=False, s2_clear_pct=0.0)
    w2 = d2 = np.zeros_like(w1)
    if C["use_optical"]:
        try:
            fd = dt.date.fromisoformat(C["flood_date"])
            s2pre = s2_composite(grid, fd - dt.timedelta(days=45), fd - dt.timedelta(days=1))
            s2post = s2_composite(grid, fd + dt.timedelta(days=1), fd + dt.timedelta(days=20))
            w2, d2, frac = detect_s2(s2pre, s2post, slope, hand, C)
            info.update(s2_clear_pct=round(100 * frac, 1), s2_used=frac > 0.15)
            if frac <= 0.15: w2 = d2 = np.zeros_like(w1)
        except Exception as e:
            print("Optical step skipped:", e)
    water = w1 | w2
    debris = (d1 | d2) & ~water
    cls = np.zeros(slope.shape, "uint8"); cls[debris] = 2; cls[water] = 1
    return dict(cls=cls, dem=dem, slope=slope, hand=hand, perm_water=perm, info=info)

def vectorize(cls, grid):
    rows = []
    for v, name in [(1, "water"), (2, "debris")]:
        m = cls == v
        for geom, _ in features.shapes(m.astype("uint8"), mask=m, transform=grid.transform):
            g = shape(geom).simplify(grid.res / 2)
            if not g.is_empty: rows.append(dict(kind=name, geometry=g))
    return gpd.GeoDataFrame(rows, geometry="geometry", crs=grid.epsg) if rows else \
           gpd.GeoDataFrame({"kind": []}, geometry=[], crs=grid.epsg)

## 5. Q2 – What was damaged? (pre-event OpenStreetMap via ohsome)

In [34]:
def ohsome(bbox, flt, time_, cols):
    r = requests.post(OHSOME_URL, data={"bboxes": ",".join(map(str, bbox)), "time": time_,
                                        "filter": flt, "properties": "tags"}, timeout=900)
    r.raise_for_status()
    fc = r.json().get("features", [])
    gdf = gpd.GeoDataFrame.from_features(fc, crs=4326) if fc else gpd.GeoDataFrame(geometry=[], crs=4326)
    for c in cols:
        if c not in gdf: gdf[c] = None
    return gdf

ROAD_CLASSES = "motorway,trunk,primary,secondary,tertiary,unclassified,residential,living_street,track," \
               "motorway_link,trunk_link,primary_link,secondary_link,tertiary_link"

def load_osm(grid, C):
    bb, t = C["bbox"], C["osm_date"]
    osm = dict(
        buildings=ohsome(bb, "building=* and geometry:polygon", t, ["building", "name"]),
        roads=ohsome(bb, f"highway in ({ROAD_CLASSES}) and geometry:line", t, ["highway", "bridge", "name"]),
        places=ohsome(bb, "place in (city,town,village,hamlet) and geometry:point", t, ["place", "name"]),
        hospitals=ohsome(bb, "(amenity=hospital or healthcare=hospital) and (geometry:point or geometry:polygon)", t, ["name"]),
    )
    if osm["hospitals"].empty:
        print("No hospitals in AOI -> falling back to clinics")
        osm["hospitals"] = ohsome(bb, "amenity=clinic and (geometry:point or geometry:polygon)", t, ["name"])
    return {k: v.to_crs(grid.epsg) for k, v in osm.items()}

def damage_assessment(osm, flood, buffer_m=30):
    fz = shapely.union_all(flood.geometry.values) if len(flood) else shapely.Polygon()
    near = fz.buffer(buffer_m)
    b, r = osm["buildings"].copy(), osm["roads"].copy()
    b["hit"] = b.geometry.intersects(fz).values if len(b) else []
    b["at_risk"] = (~b["hit"]) & b.geometry.intersects(near).values if len(b) else []
    r["hit"] = r.geometry.intersects(fz).values if len(r) else []
    r["near"] = (~r["hit"]) & r.geometry.intersects(near).values if len(r) else []
    r["is_bridge"] = r["bridge"].notna() & (r["bridge"].astype(str) != "no")
    r["len_hit_m"] = [g.intersection(fz).length if h else 0.0 for g, h in zip(r.geometry, r["hit"])] if len(r) else []
    br = r[r.is_bridge]
    stats = dict(buildings_total=int(len(b)), buildings_hit=int(b["hit"].sum()), buildings_at_risk=int(b["at_risk"].sum()),
                 road_km_total=round(float(r.length.sum()) / 1000, 1), road_km_hit=round(float(r["len_hit_m"].sum()) / 1000, 2),
                 bridges_total=int(len(br)), bridges_hit=int(br["hit"].sum()), bridges_at_risk=int(br["near"].sum()))
    return b, r, fz, stats

## 6. Q3 – Who is cut off? (road graph before vs after)

In [35]:
def build_graph(roads):
    G = nx.Graph()
    for geom in roads.geometry:
        for line in (geom.geoms if geom.geom_type == "MultiLineString" else [geom]):
            c = [(round(x, 1), round(y, 1)) for x, y in line.coords]
            for a, b in zip(c[:-1], c[1:]):
                if a != b:
                    G.add_edge(a, b, length=math.dist(a, b), seg=LineString([a, b]))
    return G

def remove_flooded_edges(G, fz):
    G2 = G.copy()
    ed = list(G2.edges(data=True))
    if not ed or fz.is_empty: return G2
    segs = np.array([d["seg"] for _, _, d in ed], dtype=object)
    hit = shapely.intersects(segs, fz)
    G2.remove_edges_from([(u, v) for (u, v, _), h in zip(ed, hit) if h])
    return G2

def cutoff_analysis(G_pre, G_post, osm, fz, snap_tol=500):
    nodes = np.array(list(G_pre.nodes)); tree = cKDTree(nodes)
    def snap(gdf):
        c = gdf.geometry.centroid
        d, i = tree.query(np.c_[c.x, c.y])
        return [tuple(nodes[j]) if dd <= snap_tol else None for dd, j in zip(d, i)]
    pl = osm["places"].copy()
    pl["node"] = snap(pl)
    hosp = [n for n in snap(osm["hospitals"]) if n is not None] if len(osm["hospitals"]) else []
    towns = osm["places"][osm["places"]["place"].isin(["town", "city"])]
    town_nodes = [n for n in snap(towns) if n is not None] if len(towns) else []
    def dists(G, src):
        return nx.multi_source_dijkstra_path_length(G, src, weight="length") if src else {}
    D = {k: (dists(G_pre, s), dists(G_post, s)) for k, s in [("hospital", hosp), ("town", town_nodes)]}
    in_flood = pl.geometry.intersects(fz.buffer(100)).values if len(pl) else []
    pl["in_flood"] = in_flood
    for k, (pre, post) in D.items():
        stat, dpre, dpost = [], [], []
        for n in pl["node"]:
            a = pre.get(n) if n else None; b = post.get(n) if n else None
            dpre.append(None if a is None else round(a / 1000, 1)); dpost.append(None if b is None else round(b / 1000, 1))
            if n is None: stat.append("no_road_data")
            elif a is None: stat.append("not_connected_before")
            elif b is None: stat.append("CUT_OFF")
            elif b > 1.5 * a + 500: stat.append("detour")
            else: stat.append("ok")
        pl[f"{k}_status"], pl[f"{k}_km_pre"], pl[f"{k}_km_post"] = stat, dpre, dpost
    return pl

## 7. Bonus – trace the flood path downstream from an upstream point (Copernicus DEM)

In [36]:
def trace_flood_path(dem, grid, lonlat, places, corridor_m=500, max_steps=40000):
    x, y = grid.from_ll.transform(*lonlat)
    c = int((x - grid.bounds[0]) / grid.res); r = int((grid.bounds[3] - y) / grid.res)
    H, W = dem.shape
    if not (0 <= r < H and 0 <= c < W): raise ValueError("Upstream point is outside the AOI")
    z = ndi.gaussian_filter(dem, 1.5)
    r0, r1, c0, c1 = max(r - 5, 0), min(r + 6, H), max(c - 5, 0), min(c + 6, W)
    sub = z[r0:r1, c0:c1]; k = np.unravel_index(np.argmin(sub), sub.shape); r, c = r0 + k[0], c0 + k[1]
    path, seen = [(r, c)], {(r, c)}
    for _ in range(max_steps):
        cand = []
        for dr in (-1, 0, 1):
            for dc in (-1, 0, 1):
                n = (r + dr, c + dc)
                if (dr or dc) and 0 <= n[0] < H and 0 <= n[1] < W and n not in seen:
                    cand.append((z[n] - z[r, c], n))
        if not cand: break
        _, (r, c) = min(cand, key=lambda t: t[0] / (1.414 if t[1][0] != r and t[1][1] != c else 1.0))
        path.append((r, c)); seen.add((r, c))
        if r in (0, H - 1) or c in (0, W - 1): break
    xy = np.array([(grid.bounds[0] + (c_ + .5) * grid.res, grid.bounds[3] - (r_ + .5) * grid.res) for r_, c_ in path])
    cum = np.r_[0, np.cumsum(np.hypot(*np.diff(xy, axis=0).T))]
    cen = places.geometry.centroid; d, i = cKDTree(xy).query(np.c_[cen.x, cen.y])
    t = places.assign(km_downstream=np.round(cum[i] / 1000, 1), dist_to_path_m=np.round(d))
    t = t[t.dist_to_path_m <= corridor_m].sort_values("km_downstream")
    return LineString(xy), t[["name", "place", "km_downstream", "dist_to_path_m"]]

## 8. Facts -> Situation-report copilot (English + Nepali)
The LLM receives ONLY a JSON of numbers computed by the system, and every number in its reply is
validated against that JSON. If validation fails twice, a deterministic template is used instead.

In [37]:
NUM = re.compile(r"\d+(?:\.\d+)?")
MODEL = "claude-sonnet-5-5"

def build_facts(C, info, stats, flood_area, places, path_table):
    names = lambda col: sorted(places.loc[places[col] == "CUT_OFF", "name"].dropna().tolist())
    return {
        "event_date": C["flood_date"], "osm_snapshot": C["osm_date"],
        "imagery": info,
        "flood_area_km2": flood_area,
        "damage": stats,
        "settlements": {"total": int(len(places)),
                        "cut_off_from_hospital": names("hospital_status"),
                        "cut_off_from_town": names("town_status"),
                        "detour_to_hospital": sorted(places.loc[places["hospital_status"] == "detour", "name"].dropna().tolist()),
                        "inside_flood_zone": sorted(places.loc[places["in_flood"], "name"].dropna().tolist())},
        "flood_path_settlements": [] if path_table is None else path_table.fillna("").to_dict("records"),
        "limitations": ["Sentinel satellites revisit every few days; no early warning.",
                        "Damage is inferred from overlap with the flood mask, not field-verified.",
                        "OSM is incomplete in remote areas; roads missing from OSM cannot be analysed."],
    }

def numbers_ok(text, facts):
    allowed = {float(x) for x in NUM.findall(json.dumps(facts, ensure_ascii=False))}
    txt = re.sub(r"(?<=\d),(?=\d{3})", "", text)
    bad = [x for x in NUM.findall(txt) if float(x) not in allowed]
    return bad

def template_report(f, lang="en"):
    d, s = f["damage"], f["settlements"]
    if lang == "ne":
        return (f"बाढी मिति {f['event_date']}। उपग्रह आधारित अनुमान: प्रभावित क्षेत्र {f['flood_area_km2']} वर्ग किमी। "
                f"भवन प्रभावित {d['buildings_hit']} (जोखिममा {d['buildings_at_risk']}), सडक {d['road_km_hit']} किमी, "
                f"पुल प्रभावित {d['bridges_hit']}। अस्पतालसँग सडक सम्पर्क टुटेका बस्ती: "
                f"{', '.join(s['cut_off_from_hospital']) or 'कुनै छैन'}। यो शैक्षिक प्रोटोटाइप हो; क्षेत्र जाँच आवश्यक छ।")
    return (f"Flood date {f['event_date']}. Satellite-derived estimate: flooded/debris area {f['flood_area_km2']} km2. "
            f"Buildings hit {d['buildings_hit']} (at risk {d['buildings_at_risk']}); road length hit {d['road_km_hit']} km; "
            f"bridges hit {d['bridges_hit']}. Settlements cut off from a hospital: "
            f"{', '.join(s['cut_off_from_hospital']) or 'none detected'}. Educational prototype; field verification required.")

SYSTEM = ("You are a situation-report assistant for disaster-response teams. Use ONLY the JSON FACTS provided. "
          "Never invent, estimate, round, sum or derive any number: copy numbers exactly as they appear in FACTS. "
          "If FACTS do not contain the answer, say so. State that results are satellite-derived estimates from an "
          "educational prototype. Do not number list items. Use Western digits 0-9 even in Nepali. "
          "Be concise, factual and respectful; never speculate about victims.")

def llm(question, facts, lang):
    import anthropic
    cl = anthropic.Anthropic(api_key=ANTHROPIC_KEY)
    language = {"en": "English", "ne": "Nepali (Devanagari script)"}[lang]
    msg = f"FACTS:\n{json.dumps(facts, ensure_ascii=False)}\n\nRespond in {language}.\nTASK: {question}"
    for attempt in range(3):
        r = cl.messages.create(model=MODEL, max_tokens=900, system=SYSTEM, messages=[{"role": "user", "content": msg}])
        text = "".join(b.text for b in r.content if b.type == "text")
        bad = numbers_ok(text, facts)
        if not bad: return text
        msg += f"\n\nYour previous answer contained numbers not in FACTS: {bad}. Rewrite using only numbers from FACTS."
    return None

def ask(question, facts, lang="en"):
    """Chat with the copilot. Falls back to a template for report requests if no key / validation fails."""
    if not ANTHROPIC_KEY:
        return "(No ANTHROPIC_API_KEY set)\n" + template_report(facts, lang)
    out = llm(question, facts, lang)
    return out or ("(LLM output failed number validation; using template)\n" + template_report(facts, lang))

def situation_report(facts):
    q = ("Write a one-page situation report with sections: Summary, Flood extent, Damage, Cut-off settlements, "
         "Flood-path settlements (if any), Limitations. Max 250 words.")
    return {lang: ask(q, facts, lang) for lang in ("en", "ne")}

### Copilot LLM = Groq (free). Overrides the earlier llm() function.

In [38]:
GROQ_MODEL = "llama-3.3-70b-versatile"   # change if Groq retires it

def llm(question, facts, lang):
    language = {"en": "English", "ne": "Nepali (Devanagari script)"}[lang]
    msg = f"FACTS:\n{json.dumps(facts, ensure_ascii=False)}\n\nRespond in {language}.\nTASK: {question}"
    for attempt in range(3):
        r = requests.post("https://api.groq.com/openai/v1/chat/completions",
            headers={"Authorization": f"Bearer {GROQ_KEY}"},
            json={"model": GROQ_MODEL, "temperature": 0.2, "max_tokens": 1200,
                  "messages": [{"role": "system", "content": SYSTEM},
                               {"role": "user", "content": msg}]}, timeout=120)
        if r.status_code != 200:
            print("Groq error:", r.status_code, r.text[:300]); return None
        text = r.json()["choices"][0]["message"]["content"]
        bad = numbers_ok(text, facts)
        if not bad: return text
        msg += f"\n\nYour previous answer contained numbers not in FACTS: {bad}. Rewrite using only numbers from FACTS."
    return None

## 9. Dashboard + report outputs

In [39]:
ATTR = ("Contains modified Copernicus Sentinel data 2026. Produced using Copernicus WorldDEM-30 © DLR e.V. 2010–2014 and "
        "© Airbus Defence and Space GmbH 2014–2018 provided under COPERNICUS by the European Union and ESA; all rights reserved. "
        "© OpenStreetMap contributors.")

def ll(gdf): return gdf.to_crs(4326)

def make_dashboard(grid, C, flood, b, r, places, hosp, flood_line, path):
    cx, cy = grid.to_ll.transform((grid.bounds[0] + grid.bounds[2]) / 2, (grid.bounds[1] + grid.bounds[3]) / 2)
    m = folium.Map(location=[cy, cx], zoom_start=11, tiles="CartoDB positron")
    color = {"water": "#1f78b4", "debris": "#8c5a2b"}
    for kind in ("water", "debris"):
        g = flood[flood.kind == kind]
        if len(g):
            folium.GeoJson(ll(g).to_json(), name=f"Flood: {kind}",
                           style_function=lambda _, c=color[kind]: dict(color=c, fillColor=c, fillOpacity=.55, weight=1)).add_to(m)
    bh = b[b.hit | b.at_risk]
    if len(bh):
        folium.GeoJson(ll(bh[["hit", "at_risk", "geometry"]]).to_json(), name="Buildings hit / at risk",
                       style_function=lambda f: dict(color="red" if f["properties"]["hit"] else "orange", weight=1, fillOpacity=.7)).add_to(m)
    rh = r[r.hit]
    if len(rh):
        folium.GeoJson(ll(rh[["highway", "is_bridge", "geometry"]]).to_json(), name="Roads/bridges hit",
                       style_function=lambda f: dict(color="purple" if f["properties"]["is_bridge"] else "red", weight=4)).add_to(m)
    fg = folium.FeatureGroup(name="Settlements").add_to(m)
    for _, p in ll(places).iterrows():
        st = p["hospital_status"]
        col = {"CUT_OFF": "red", "detour": "orange", "ok": "green"}.get(st, "gray")
        folium.CircleMarker([p.geometry.y, p.geometry.x], radius=7, color=col, fill=True, fill_opacity=.9,
                            tooltip=f"{p['name']} | hospital: {st} | town: {p['town_status']}").add_to(fg)
    fh = folium.FeatureGroup(name="Hospitals").add_to(m)
    for _, h in ll(hosp).iterrows():
        pt = h.geometry.centroid
        folium.Marker([pt.y, pt.x], icon=folium.Icon(color="black", icon="plus"), tooltip=h.get("name") or "hospital").add_to(fh)
    if flood_line is not None:
        folium.GeoJson(gpd.GeoSeries([flood_line], crs=grid.epsg).to_crs(4326).to_json(), name="Traced flood path",
                       style_function=lambda _: dict(color="cyan", weight=3, dashArray="6")).add_to(m)
    folium.LayerControl().add_to(m)
    m.get_root().html.add_child(folium.Element(
        f"<div style='position:fixed;bottom:0;left:0;right:0;z-index:9999;background:#fffc;font:11px sans-serif;padding:3px'>"
        f"{ATTR} Educational prototype – not an operational tool.</div>"))
    m.save(path)
    return m

def make_report_html(facts, texts, png_b64, path):
    esc = lambda s: s.replace("&", "&amp;").replace("<", "&lt;").replace("\n", "<br>")
    d = facts["damage"]
    html = f"""<!doctype html><html><head><meta charset='utf-8'><title>Situation report</title>
<style>body{{font:12px/1.4 'Noto Sans','Noto Sans Devanagari',sans-serif;max-width:800px;margin:auto}}
td,th{{border:1px solid #999;padding:2px 6px}}table{{border-collapse:collapse}}img{{max-width:100%}}
.f{{font-size:9px;color:#555}}</style></head><body>
<h2>Flood situation report – {facts['event_date']}</h2>
<img src='data:image/png;base64,{png_b64}'>
<table><tr><th>Flooded/debris area</th><th>Buildings hit</th><th>At risk</th><th>Road km hit</th><th>Bridges hit</th></tr>
<tr><td>{facts['flood_area_km2']} km²</td><td>{d['buildings_hit']}</td><td>{d['buildings_at_risk']}</td><td>{d['road_km_hit']}</td><td>{d['bridges_hit']}</td></tr></table>
<h3>English</h3><p>{esc(texts['en'])}</p><h3>नेपाली</h3><p>{esc(texts['ne'])}</p>
<p class='f'>{ATTR}<br>Educational prototype; not an operational tool. Estimates are not field-verified.</p></body></html>"""
    open(path, "w", encoding="utf-8").write(html)

def static_png(grid, cls, perm):
    rgb = np.ones(cls.shape + (3,)); rgb[perm] = (.75, .85, .95)
    rgb[cls == 1] = (.12, .47, .71); rgb[cls == 2] = (.55, .35, .17)
    fig, ax = plt.subplots(figsize=(6, 5)); ax.imshow(rgb); ax.axis("off")
    ax.set_title("Water (blue) / debris-change (brown)")
    buf = io.BytesIO(); fig.savefig(buf, format="png", dpi=110, bbox_inches="tight"); plt.close(fig)
    return base64.b64encode(buf.getvalue()).decode()

## 10. Evaluation against EMSR927 (CHECKING ONLY – never use as input)

In [40]:
def evaluate_against_reference(flood, ref_path, grid):
    """ref_path: GeoJSON/Shapefile of the official EMS flood/debris extent you downloaded manually."""
    ref = gpd.read_file(ref_path).to_crs(grid.epsg)
    rast = lambda g: features.rasterize([(x, 1) for x in g.geometry if x is not None], out_shape=(grid.H, grid.W),
                                        transform=grid.transform, dtype="uint8").astype(bool)
    p, t = rast(flood), rast(ref)
    tp = (p & t).sum(); fp = (p & ~t).sum(); fn = (~p & t).sum()
    res = dict(IoU=tp / max(tp + fp + fn, 1), precision=tp / max(tp + fp, 1), recall=tp / max(tp + fn, 1))
    res["F1"] = 2 * res["precision"] * res["recall"] / max(res["precision"] + res["recall"], 1e-9)
    return {k: round(float(v), 3) for k, v in res.items()}

In [41]:
from rasterio.warp import reproject, Resampling

def get_dem(grid):
    """Copernicus DEM GLO-30 from the public AWS bucket (no key), resampled onto our grid."""
    w, s, e, n = CONFIG["bbox"]
    dem = np.full((grid.H, grid.W), np.nan, "float32")
    with rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR"):
        for lat in range(math.floor(s), math.floor(n) + 1):
            for lon in range(math.floor(w), math.floor(e) + 1):
                ns = f"N{lat:02d}" if lat >= 0 else f"S{-lat:02d}"
                ew = f"E{lon:03d}" if lon >= 0 else f"W{-lon:03d}"
                name = f"Copernicus_DSM_COG_10_{ns}_00_{ew}_00_DEM"
                url = f"https://copernicus-dem-30m.s3.amazonaws.com/{name}/{name}.tif"
                try:
                    with rasterio.open(url) as src:
                        tmp = np.full(dem.shape, np.nan, "float32")
                        reproject(rasterio.band(src, 1), tmp, dst_transform=grid.transform,
                                  dst_crs=f"EPSG:{grid.epsg}", dst_nodata=np.nan,
                                  resampling=Resampling.bilinear)
                    dem = np.where(np.isnan(dem), tmp, dem)
                except Exception as ex:
                    print("DEM tile skipped:", name, ex)
    if np.isnan(dem).all():
        raise RuntimeError("No DEM tiles could be read - check internet access / AOI")
    dem[np.isnan(dem)] = np.nanmedian(dem)
    return dem

In [42]:
from shapely.geometry import Point, Polygon

UA = {"User-Agent": "floodwatch-hackathon/1.0 (educational prototype)"}
OVERPASS = ["https://overpass-api.de/api/interpreter",
            "https://overpass.kumi.systems/api/interpreter"]

def ohsome(bbox, flt, time_, cols):
    r = requests.post(OHSOME_URL, headers=UA, data={"bboxes": ",".join(map(str, bbox)), "time": time_,
                      "filter": flt, "properties": "tags"}, timeout=900)
    r.raise_for_status()
    fc = r.json().get("features", [])
    gdf = gpd.GeoDataFrame.from_features(fc, crs=4326) if fc else gpd.GeoDataFrame(geometry=[], crs=4326)
    for c in cols:
        if c not in gdf: gdf[c] = None
    return gdf

def overpass(bbox, body, date, cols, geom):
    w, s, e, n = bbox
    q = (f'[out:json][timeout:600][date:"{date}T00:00:00Z"];({body.replace("@@", f"({s},{w},{n},{e})")});'
         + ("out center tags;" if geom == "center" else "out geom tags;"))
    last = None
    for url in OVERPASS:
        try:
            r = requests.post(url, data={"data": q}, headers=UA, timeout=900)
            if r.status_code == 200:
                js = r.json(); break
            last = f"{url} -> {r.status_code}"
        except Exception as ex:
            last = f"{url} -> {ex}"
    else:
        raise RuntimeError("All Overpass servers failed: " + str(last))
    rows = []
    for el in js.get("elements", []):
        tags = el.get("tags", {}); g = None
        if el["type"] == "node" and "lat" in el: g = Point(el["lon"], el["lat"])
        elif geom == "center" and "center" in el: g = Point(el["center"]["lon"], el["center"]["lat"])
        elif el["type"] == "way" and "geometry" in el:
            pts = [(p["lon"], p["lat"]) for p in el["geometry"]]
            if geom == "poly":
                if len(pts) >= 4 and pts[0] == pts[-1]: g = Polygon(pts)
            elif len(pts) >= 2: g = LineString(pts)
        if g is not None:
            rows.append({**{c: tags.get(c) for c in cols}, "geometry": g})
    return gpd.GeoDataFrame(rows, geometry="geometry", crs=4326) if rows else \
           gpd.GeoDataFrame({c: [] for c in cols}, geometry=[], crs=4326)

def load_osm(grid, C):
    bb, t = C["bbox"], C["osm_date"]
    rc = ROAD_CLASSES.replace(",", "|")
    L = {
     "buildings": ("building=* and geometry:polygon", ["building", "name"], 'way["building"]@@;', "poly"),
     "roads": (f"highway in ({ROAD_CLASSES}) and geometry:line", ["highway", "bridge", "name"],
               f'way["highway"~"^({rc})$"]@@;', "line"),
     "places": ("place in (city,town,village,hamlet) and geometry:point", ["place", "name"],
                'node["place"~"^(city|town|village|hamlet)$"]@@;', "point"),
     "hospitals": ("(amenity=hospital or healthcare=hospital) and (geometry:point or geometry:polygon)", ["name"],
                   'nwr["amenity"="hospital"]@@;nwr["healthcare"="hospital"]@@;', "center"),
    }
    osm = {}
    for k, (flt, cols, op, geom) in L.items():
        try:
            osm[k] = ohsome(bb, flt, t, cols); print(k, "via ohsome:", len(osm[k]))
        except Exception as ex:
            print(k, "ohsome failed (", str(ex)[:60], ") -> Overpass")
            osm[k] = overpass(bb, op, t, cols, geom); print(k, "via Overpass:", len(osm[k]))
    if osm["hospitals"].empty:
        print("No hospitals in AOI -> falling back to clinics")
        osm["hospitals"] = overpass(bb, 'nwr["amenity"="clinic"]@@;', t, ["name"], "center")
    return {k: v.to_crs(grid.epsg) for k, v in osm.items()}

In [43]:
UA = {"User-Agent": "floodwatch-hackathon/1.0 (educational prototype)"}
OVERPASS = ["https://overpass-api.de/api/interpreter",
            "https://overpass.kumi.systems/api/interpreter",
            "https://overpass.private.coffee/api/interpreter"]

# remember the flood polygons so buildings are only fetched near them
_orig_vectorize = vectorize
def vectorize(cls, grid):
    g = _orig_vectorize(cls, grid)
    globals()["_LAST_FLOOD"] = g
    return g

def op_fetch(bb, body, date, cols, geom):
    w, s, e, n = bb
    q = (f'[out:json][timeout:180][date:"{date}T00:00:00Z"];({body.replace("@@", f"({s},{w},{n},{e})")});'
         + ("out center tags;" if geom == "center" else "out geom tags;"))
    for attempt in range(3):
        for url in OVERPASS:
            try:
                r = requests.post(url, data={"data": q}, headers=UA, timeout=240)
                if r.status_code == 200:
                    rows = []
                    for el in r.json().get("elements", []):
                        tags = el.get("tags", {}); g = None
                        if el["type"] == "node" and "lat" in el: g = Point(el["lon"], el["lat"])
                        elif geom == "center" and "center" in el: g = Point(el["center"]["lon"], el["center"]["lat"])
                        elif el["type"] == "way" and "geometry" in el:
                            pts = [(p["lon"], p["lat"]) for p in el["geometry"]]
                            if geom == "poly":
                                if len(pts) >= 4 and pts[0] == pts[-1]: g = Polygon(pts)
                            elif len(pts) >= 2: g = LineString(pts)
                        if g is not None:
                            rows.append({**{c: tags.get(c) for c in cols}, "osmid": f'{el["type"]}/{el["id"]}', "geometry": g})
                    return rows
                print("   ", url.split("/")[2], r.status_code)
            except Exception as ex:
                print("   ", url.split("/")[2], str(ex)[:60])
        time.sleep(5 * (attempt + 1))
    raise RuntimeError("Overpass failed for a tile after retries")

def make_tiles(bb, step):
    w, s, e, n = bb
    xs = np.arange(w, e, step); ys = np.arange(s, n, step)
    return [(x, y, min(x + step, e), min(y + step, n)) for x in xs for y in ys]

def op_layer(tiles, body, date, cols, geom):
    rows = []
    for i, tb in enumerate(tiles):
        rows += op_fetch(tb, body, date, cols, geom)
        print(f"   tile {i+1}/{len(tiles)} ok, total {len(rows)}")
        time.sleep(1)
    gdf = gpd.GeoDataFrame(rows, geometry="geometry", crs=4326) if rows else \
          gpd.GeoDataFrame({c: [] for c in cols + ["osmid"]}, geometry=[], crs=4326)
    return gdf.drop_duplicates("osmid") if len(gdf) else gdf

def load_osm(grid, C):
    bb, t = C["bbox"], C["osm_date"]
    rc = ROAD_CLASSES.replace(",", "|")
    try:   # quick test whether ohsome accepts us
        ohsome((85.3, 28.1, 85.31, 28.11), "place=*", t, ["name"]); use_ohsome = True
    except Exception as ex:
        print("ohsome unavailable (", str(ex)[:50], ") -> using Overpass"); use_ohsome = False
    big = make_tiles(bb, 0.12)
    osm = {}
    # roads, places, hospitals: whole area
    if use_ohsome:
        osm["roads"] = ohsome(bb, f"highway in ({ROAD_CLASSES}) and geometry:line", t, ["highway", "bridge", "name"])
        osm["places"] = ohsome(bb, "place in (city,town,village,hamlet) and geometry:point", t, ["place", "name"])
        osm["hospitals"] = ohsome(bb, "(amenity=hospital or healthcare=hospital) and (geometry:point or geometry:polygon)", t, ["name"])
    else:
        print("roads..."); osm["roads"] = op_layer(big, f'way["highway"~"^({rc})$"]@@;', t, ["highway", "bridge", "name"], "line")
        print("places..."); osm["places"] = op_layer(big, 'node["place"~"^(city|town|village|hamlet)$"]@@;', t, ["place", "name"], "point")
        print("hospitals..."); osm["hospitals"] = op_layer(big, 'nwr["amenity"="hospital"]@@;nwr["healthcare"="hospital"]@@;', t, ["name"], "center")
    if osm["hospitals"].empty:
        print("No hospitals in AOI -> trying clinics")
        osm["hospitals"] = op_layer(big, 'nwr["amenity"="clinic"]@@;', t, ["name"], "center")
    # buildings: only tiles close to the flood zone
    fl = globals().get("_LAST_FLOOD")
    if fl is None or len(fl) == 0:
        print("No flood polygons -> skipping buildings"); osm["buildings"] = gpd.GeoDataFrame({"building": [], "name": []}, geometry=[], crs=4326)
    else:
        zone = shapely.union_all(fl.to_crs(4326).geometry.values).buffer(0.01)
        small = [tb for tb in make_tiles(bb, 0.03) if box(*tb).intersects(zone)][:60]
        print(f"buildings: {len(small)} tiles near flood zone...")
        if use_ohsome:
            osm["buildings"] = ohsome(bb, "building=* and geometry:polygon", t, ["building", "name"])
        else:
            osm["buildings"] = op_layer(small, 'way["building"]@@;', t, ["building", "name"], "poly")
    for k, v in osm.items(): print(f"  {k}: {len(v)}")
    return {k: v.to_crs(grid.epsg) for k, v in osm.items()}

In [44]:
# smaller area = fewer tiles (adjust if you want)
CONFIG["bbox"] = (85.25, 28.05, 85.45, 28.30)

_OPCACHE = globals().get("_OPCACHE", {})
FAILED_TILES = []

def _op_once(tb, body, date, cols, geom, tries=2):
    key = (body, date, tuple(round(x, 4) for x in tb), geom)
    if key in _OPCACHE: return _OPCACHE[key]
    w, s, e, n = tb
    q = (f'[out:json][timeout:120][date:"{date}T00:00:00Z"];({body.replace("@@", f"({s},{w},{n},{e})")});'
         + ("out center tags;" if geom == "center" else "out geom tags;"))
    for attempt in range(tries):
        for url in OVERPASS:
            try:
                r = requests.post(url, data={"data": q}, headers=UA, timeout=150)
                if r.status_code == 200:
                    rows = []
                    for el in r.json().get("elements", []):
                        tags = el.get("tags", {}); g = None
                        if el["type"] == "node" and "lat" in el: g = Point(el["lon"], el["lat"])
                        elif geom == "center" and "center" in el: g = Point(el["center"]["lon"], el["center"]["lat"])
                        elif el["type"] == "way" and "geometry" in el:
                            pts = [(p["lon"], p["lat"]) for p in el["geometry"]]
                            if geom == "poly":
                                if len(pts) >= 4 and pts[0] == pts[-1]: g = Polygon(pts)
                            elif len(pts) >= 2: g = LineString(pts)
                        if g is not None:
                            rows.append({**{c: tags.get(c) for c in cols}, "osmid": f'{el["type"]}/{el["id"]}', "geometry": g})
                    _OPCACHE[key] = rows
                    return rows
            except Exception:
                pass
        time.sleep(8 * (attempt + 1))
    return None

def op_layer(tiles, body, date, cols, geom):
    rows, pending = [], []
    for i, tb in enumerate(tiles):
        r = _op_once(tb, body, date, cols, geom)
        if r is None: pending.append(tb); print(f"   tile {i+1}/{len(tiles)} busy, will retry later")
        else: rows += r; print(f"   tile {i+1}/{len(tiles)} ok, total {len(rows)}")
        time.sleep(1)
    for rnd in range(2):                      # retry failed tiles after the servers calm down
        if not pending: break
        print(f"   retrying {len(pending)} tile(s) in 30 s ..."); time.sleep(30)
        still = []
        for tb in pending:
            r = _op_once(tb, body, date, cols, geom, tries=3)
            if r is None: still.append(tb)
            else: rows += r; print("   retried tile ok, total", len(rows))
        pending = still
    if pending:
        FAILED_TILES.extend(pending)
        print(f"   WARNING: {len(pending)} tile(s) missing -> coverage incomplete (mention in limitations)")
    gdf = gpd.GeoDataFrame(rows, geometry="geometry", crs=4326) if rows else \
          gpd.GeoDataFrame({c: [] for c in cols + ["osmid"]}, geometry=[], crs=4326)
    return gdf.drop_duplicates("osmid") if len(gdf) else gdf

_mt = make_tiles
def make_tiles(bb, step): return _mt(bb, min(step, 0.06))
from shapely.geometry import box

In [45]:
# ---- PATCHES (override earlier definitions) ----
def s1_db(arr):
    """(VV, VH, mask) -> speckle-reduced dB. Non-finite/absurd values excluded BEFORE filtering."""
    vv, vh, m = arr
    valid = ((m > 0) & np.isfinite(vv) & np.isfinite(vh)
             & (vv > 0) & (vh > 0) & (vv < 10) & (vh < 10))
    den = ndi.uniform_filter(valid.astype("float64"), 5)
    out = []
    for b in (vv, vh):
        x = np.where(valid, b, 0.0).astype("float64")
        num = ndi.uniform_filter(x, 5)
        lin = np.where(den > 0.3, num / np.maximum(den, 1e-6), np.nan)
        out.append(10 * np.log10(np.clip(lin, 1e-5, None)))
    return out[0], out[1]

GROQ_MODEL = "openai/gpt-oss-120b"

def llm(question, facts, lang):
    language = {"en": "English", "ne": "Nepali (Devanagari script)"}[lang]
    msg = f"FACTS:\n{json.dumps(facts, ensure_ascii=False)}\n\nRespond in {language}.\nTASK: {question}"
    for attempt in range(3):
        r = requests.post("https://api.groq.com/openai/v1/chat/completions",
            headers={"Authorization": f"Bearer {GROQ_KEY}"},
            json={"model": GROQ_MODEL, "temperature": 0.2, "max_tokens": 3000,
                  "messages": [{"role": "system", "content": SYSTEM},
                               {"role": "user", "content": msg}]}, timeout=120)
        if r.status_code != 200:
            print("Groq error:", r.status_code, r.text[:300]); return None
        text = r.json()["choices"][0]["message"].get("content") or ""
        if not text.strip(): continue
        bad = numbers_ok(text, facts)
        if not bad: return text
        msg += f"\n\nYour previous answer contained numbers not in FACTS: {bad}. Rewrite using only numbers from FACTS."
    return None

In [46]:
import heapq

def fill_depressions(dem, eps=1e-3):
    H, W = dem.shape
    z = dem.astype("float64").copy()
    closed = np.zeros((H, W), bool); heap = []
    for r in range(H):
        for c in (0, W - 1):
            heapq.heappush(heap, (z[r, c], r, c)); closed[r, c] = True
    for c in range(1, W - 1):
        for r in (0, H - 1):
            heapq.heappush(heap, (z[r, c], r, c)); closed[r, c] = True
    nb = [(-1,-1),(-1,0),(-1,1),(0,-1),(0,1),(1,-1),(1,0),(1,1)]
    while heap:
        h, r, c = heapq.heappop(heap)
        for dr, dc in nb:
            rr, cc = r + dr, c + dc
            if 0 <= rr < H and 0 <= cc < W and not closed[rr, cc]:
                closed[rr, cc] = True
                if z[rr, cc] <= h: z[rr, cc] = h + eps
                heapq.heappush(heap, (z[rr, cc], rr, cc))
    return z

def trace_flood_path(dem, grid, lonlat, places, corridor_m=500, max_steps=200000):
    x, y = grid.from_ll.transform(*lonlat)
    c = int((x - grid.bounds[0]) / grid.res); r = int((grid.bounds[3] - y) / grid.res)
    H, W = dem.shape
    if not (0 <= r < H and 0 <= c < W): raise ValueError("Upstream point is outside the AOI")
    z = fill_depressions(ndi.gaussian_filter(dem, 1.0))
    r0, r1, c0, c1 = max(r-10, 0), min(r+11, H), max(c-10, 0), min(c+11, W)
    sub = z[r0:r1, c0:c1]; k = np.unravel_index(np.argmin(sub), sub.shape); r, c = r0 + k[0], c0 + k[1]
    path = [(r, c)]
    for _ in range(max_steps):
        best, bs = None, 0.0
        for dr in (-1, 0, 1):
            for dc in (-1, 0, 1):
                if not (dr or dc): continue
                rr, cc = r + dr, c + dc
                if 0 <= rr < H and 0 <= cc < W:
                    s = (z[r, c] - z[rr, cc]) / (1.414 if dr and dc else 1.0)
                    if s > bs: bs, best = s, (rr, cc)
        if best is None: break
        r, c = best; path.append(best)
        if r in (0, H-1) or c in (0, W-1): break
    xy = np.array([(grid.bounds[0] + (c_ + .5) * grid.res, grid.bounds[3] - (r_ + .5) * grid.res) for r_, c_ in path])
    cum = np.r_[0, np.cumsum(np.hypot(*np.diff(xy, axis=0).T))]
    cen = places.geometry.centroid; d, i = cKDTree(xy).query(np.c_[cen.x, cen.y])
    t = places.assign(km_downstream=np.round(cum[i] / 1000, 1), dist_to_path_m=np.round(d))
    t = t[t.dist_to_path_m <= corridor_m].sort_values("km_downstream")
    return LineString(xy), t[["name", "place", "km_downstream", "dist_to_path_m"]]

In [47]:
def hand_proxy(dem, seeds):
    return np.zeros_like(dem)   # HAND is unreliable in this gorge; rely on slope + radar change instead

## 11. Run everything end-to-end

In [59]:
def run(C=CONFIG):
    grid = Grid(C["bbox"], C["res"]); print(f"Grid {grid.W}x{grid.H} @ {grid.res:.0f} m, EPSG:{grid.epsg}")
    F = map_flood(grid, C)
    if F["info"]["s1_valid_pct"] < 50:
        raise RuntimeError(f"Sentinel-1 coverage too low ({F['info']['s1_valid_pct']}%).")
    flood = vectorize(F["cls"], grid)
    flood_area = round(float((F["cls"] > 0).sum() * grid.res ** 2 / 1e6), 2)
    osm = load_osm(grid, C)
    b, r, fz, stats = damage_assessment(osm, flood)
    G0 = build_graph(r); G1 = remove_flooded_edges(G0, fz)
    places = cutoff_analysis(G0, G1, osm, fz)
    line, ptable = None, None
    if C.get("upstream_point"):
        try: line, ptable = trace_flood_path(F["dem"], grid, C["upstream_point"], osm["places"])
        except Exception as e: print("Flood-path step skipped:", e)
    facts = build_facts(C, F["info"], stats, flood_area, places, ptable)
    texts = situation_report(facts)
    out = C["out_dir"]
    m = make_dashboard(grid, C, flood, b, r, places, osm["hospitals"], line, f"{out}/dashboard.html")
    make_report_html(facts, texts, static_png(grid, F["cls"], F["perm_water"]), f"{out}/situation_report.html")
    json.dump(facts, open(f"{out}/facts.json", "w"), ensure_ascii=False, indent=1)
    flood.to_crs(4326).to_file(f"{out}/flood.geojson", driver="GeoJSON")
    print(json.dumps(facts["damage"], indent=1)); print(texts["en"])
    return dict(grid=grid, flood=flood, facts=facts, places=places, osm=osm, map=m)

RESULT = run()

Grid 668x933 @ 30 m, EPSG:32645
S1 pair: {'pre_day': '2026-08-16', 'post_day': '2026-08-28', 'orbit': 'ASCENDING', 'rel': 85}


ohsome unavailable ( 403 Client Error: Forbidden for url: https://api.o ) -> using Overpass
roads...
   tile 1/20 ok, total 13
   tile 2/20 ok, total 47
   tile 3/20 ok, total 83
   tile 4/20 ok, total 83
   tile 5/20 ok, total 83
   tile 6/20 ok, total 89
   tile 7/20 ok, total 127
   tile 8/20 ok, total 170
   tile 9/20 ok, total 188
   tile 10/20 ok, total 189
   tile 11/20 ok, total 189
   tile 12/20 ok, total 192
   tile 13/20 ok, total 192
   tile 14/20 ok, total 204
   tile 15/20 ok, total 205
   tile 16/20 ok, total 205
   tile 17/20 ok, total 205
   tile 18/20 ok, total 205
   tile 19/20 ok, total 205
   tile 20/20 ok, total 205
places...
   tile 1/20 ok, total 0
   tile 2/20 ok, total 10
   tile 3/20 ok, total 16
   tile 4/20 ok, total 16
   tile 5/20 ok, total 16
   tile 6/20 ok, total 16
   tile 7/20 ok, total 24
   tile 8/20 ok, total 34
   tile 9/20 ok, total 35
   tile 10/20 ok, total 35
   tile 11/20 ok, total 37
   tile 12/20 ok, total 38
   tile 13/20 ok, total 38
   

/usr/local/lib/python3.13/dist-packages/folium/raster_layers.py:130: UserWarning: CartoDB tiles now require an API key. Please provide one to continue using the tiles. You can request the key at https://carto.com/basemaps/apikey/.
  tiles = tiles.build_url(fill_subdomain=False, scale_factor="{r}")  # type: ignore


{
 "buildings_total": 2660,
 "buildings_hit": 30,
 "buildings_at_risk": 15,
 "road_km_total": 192.8,
 "road_km_hit": 0.39,
 "bridges_total": 18,
 "bridges_hit": 1,
 "bridges_at_risk": 0
}
(LLM output failed number validation; using template)
Flood date 2026-08-26. Satellite-derived estimate: flooded/debris area 0.07 km2. Buildings hit 30 (at risk 15); road length hit 0.39 km; bridges hit 1. Settlements cut off from a hospital: 热索. Educational prototype; field verification required.
Grid 668x933 @ 30 m, EPSG:32645
S1 pair: {'pre_day': '2026-08-16', 'post_day': '2026-08-28', 'orbit': 'ASCENDING', 'rel': 85}


ohsome unavailable ( 403 Client Error: Forbidden for url: https://api.o ) -> using Overpass
roads...
   tile 1/20 ok, total 13
   tile 2/20 ok, total 47
   tile 3/20 ok, total 83
   tile 4/20 ok, total 83
   tile 5/20 ok, total 83
   tile 6/20 ok, total 89
   tile 7/20 ok, total 127
   tile 8/20 ok, total 170
   tile 9/20 ok, total 188
   tile 10/20 ok, total 189
   tile 11/20 ok, total 189
   tile 12/20 ok, total 192
   tile 13/20 ok, total 192
   tile 14/20 ok, total 204
   tile 15/20 ok, total 205
   tile 16/20 ok, total 205
   tile 17/20 ok, total 205
   tile 18/20 ok, total 205
   tile 19/20 ok, total 205
   tile 20/20 ok, total 205
places...
   tile 1/20 ok, total 0
   tile 2/20 ok, total 10
   tile 3/20 ok, total 16
   tile 4/20 ok, total 16
   tile 5/20 ok, total 16
   tile 6/20 ok, total 16
   tile 7/20 ok, total 24
   tile 8/20 ok, total 34
   tile 9/20 ok, total 35
   tile 10/20 ok, total 35
   tile 11/20 ok, total 37
   tile 12/20 ok, total 38
   tile 13/20 ok, total 38
   

/usr/local/lib/python3.13/dist-packages/folium/raster_layers.py:130: UserWarning: CartoDB tiles now require an API key. Please provide one to continue using the tiles. You can request the key at https://carto.com/basemaps/apikey/.
  tiles = tiles.build_url(fill_subdomain=False, scale_factor="{r}")  # type: ignore


In [60]:
if "_bf_base" not in globals():
    _bf_base = build_facts          # capture the current version once

def build_facts(C, info, stats, flood_area, places, path_table):
    f = _bf_base(C, info, stats, flood_area, places, path_table)
    ok = ["ok", "detour", "CUT_OFF"]
    s = f["settlements"]
    s["analysed_for_hospital_access"] = int(places["hospital_status"].isin(ok).sum())
    s["no_road_data_within_500m"] = int((places["hospital_status"] == "no_road_data").sum())
    if not places["town_status"].isin(ok).any():
        s["cut_off_from_town"] = "not assessed: no town found in OSM data for this area"
    kinds = sorted(set(globals().get("_LAST_FLOOD", gpd.GeoDataFrame({"kind": []})).get("kind", [])))
    f["flood_mask_kinds"] = kinds
    if "water" not in kinds:
        f["flood_mask_note"] = "Mask contains radar-change (debris-affected) areas only; no open water was detected."
    if not any("single route" in x for x in f["limitations"]):
        f["limitations"] += [
            "Cut-off status depends on a few road segments in OpenStreetMap, which here is close to a single route; one flagged segment can isolate a whole branch.",
            "Results are sensitive to detection thresholds: more permissive settings give a much larger affected area and more cut-off settlements.",
            "No town was found in the OSM data for this area, so access to a town was not assessed.",
        ]
    return f

if "_sys_final" not in globals():
    _sys_final = True
    SYSTEM = SYSTEM + (" Report overlap with the detected mask as 'intersects the detected mask', not as impact or damage. "
                       "If a FACTS field says 'not assessed', say it was not assessed; never say 'none' for it. "
                       "Include EVERY item of FACTS limitations. Always finish the Limitations section.")

In [61]:
CONFIG.update(change_db=3.0, min_blob_px=8, max_slope_deg=25.0, max_hand_m=60.0)
CONFIG["bbox"] = (85.25, 28.05, 85.45, 28.30)
print({k: CONFIG[k] for k in ("bbox", "change_db", "min_blob_px", "max_slope_deg")})

{'bbox': (85.25, 28.05, 85.45, 28.3), 'change_db': 3.0, 'min_blob_px': 8, 'max_slope_deg': 25.0}
{'bbox': (85.25, 28.05, 85.45, 28.3), 'change_db': 3.0, 'min_blob_px': 8, 'max_slope_deg': 25.0}


In [62]:
def remove_flooded_edges(G, fz, min_len_m=30):
    G2 = G.copy()
    ed = list(G2.edges(data=True))
    if not ed or fz.is_empty: return G2
    shapely.prepare(fz)
    segs = np.array([d["seg"] for _, _, d in ed], dtype=object)
    hit = shapely.intersects(segs, fz)
    L = np.zeros(len(ed))
    L[hit] = shapely.length(shapely.intersection(segs[hit], fz))
    G2.remove_edges_from([(u, v) for (u, v, _), l in zip(ed, L) if l >= min_len_m])
    return G2

In [63]:
# ---- final report-prompt rules (guarded so re-running does not stack them) ----
if "_sys_final2" not in globals():
    _sys_final2 = True
    SYSTEM = SYSTEM + (" Make NO claims beyond what FACTS state: no statements about causes, severity, disruption or impacts. "
                       "Describe the flood area as 'flooded or debris-affected area'; say 'water' only if FACTS flood_mask_kinds contains 'water'. "
                       "Do not describe the result as complete; it is a detected minimum. "
                       "In Nepali, translate place types (village = गाउँ, hamlet = बस्ती).")

# headline settings (chosen BEFORE looking at any reference data)
CONFIG.update(change_db=3.0, min_blob_px=8, max_slope_deg=25.0, max_hand_m=60.0)
CONFIG["bbox"] = (85.25, 28.05, 85.45, 28.30)
print({k: CONFIG[k] for k in ("bbox", "change_db", "min_blob_px", "max_slope_deg")})

{'bbox': (85.25, 28.05, 85.45, 28.3), 'change_db': 3.0, 'min_blob_px': 8, 'max_slope_deg': 25.0}
{'bbox': (85.25, 28.05, 85.45, 28.3), 'change_db': 3.0, 'min_blob_px': 8, 'max_slope_deg': 25.0}


In [53]:
RESULT = run()

Grid 668x933 @ 30 m, EPSG:32645
S1 pair: {'pre_day': '2026-08-16', 'post_day': '2026-08-28', 'orbit': 'ASCENDING', 'rel': 85}


ohsome unavailable ( 403 Client Error: Forbidden for url: https://api.o ) -> using Overpass
roads...
   tile 1/20 ok, total 13
   tile 2/20 ok, total 47
   tile 3/20 ok, total 83
   tile 4/20 ok, total 83
   tile 5/20 ok, total 83
   tile 6/20 ok, total 89
   tile 7/20 ok, total 127
   tile 8/20 ok, total 170
   tile 9/20 ok, total 188
   tile 10/20 ok, total 189
   tile 11/20 ok, total 189
   tile 12/20 ok, total 192
   tile 13/20 ok, total 192
   tile 14/20 ok, total 204
   tile 15/20 ok, total 205
   tile 16/20 ok, total 205
   tile 17/20 ok, total 205
   tile 18/20 ok, total 205
   tile 19/20 ok, total 205
   tile 20/20 ok, total 205
places...
   tile 1/20 ok, total 0
   tile 2/20 ok, total 10
   tile 3/20 ok, total 16
   tile 4/20 ok, total 16
   tile 5/20 ok, total 16
   tile 6/20 ok, total 16
   tile 7/20 ok, total 24
   tile 8/20 ok, total 34
   tile 9/20 ok, total 35
   tile 10/20 ok, total 35
   tile 11/20 ok, total 37
   tile 12/20 ok, total 38
   tile 13/20 ok, total 38
   

/usr/local/lib/python3.13/dist-packages/folium/raster_layers.py:130: UserWarning: CartoDB tiles now require an API key. Please provide one to continue using the tiles. You can request the key at https://carto.com/basemaps/apikey/.
  tiles = tiles.build_url(fill_subdomain=False, scale_factor="{r}")  # type: ignore


{
 "buildings_total": 2660,
 "buildings_hit": 30,
 "buildings_at_risk": 15,
 "road_km_total": 192.8,
 "road_km_hit": 0.39,
 "bridges_total": 18,
 "bridges_hit": 1,
 "bridges_at_risk": 0
}
**Summary**  
On 2026-08-26 a flood was detected by Sentinel‑1 imagery (pre‑image 2026-08-16, post‑image 2026-08-28, orbit ASCENDING, validity 99.7 %). The flood mask shows debris‑related change only; no open water was identified. The estimated flooded area is 0.07 km². Results are satellite‑derived estimates from an educational prototype.

**Flood extent**  
The flood mask is based on radar‑change (debris) detection. Sentinel‑2 data were not used (s2_used false) and only 11.5 % of the scene was cloud‑free.

**Damage**  
Buildings: 2660 total, 30 hit, 15 at risk. Roads: 192.8 km total, 0.39 km hit. Bridges: 18 total, 1 hit, 0 at risk.

**Cut‑off settlements**  
One village, 热索, is cut off from the nearest hospital. Access to a town could not be assessed because no town was found in the OSM data. No de

## 12. Open the dashboard, chat with the copilot, evaluate

In [54]:
from IPython.display import display
display(RESULT["map"])   # interactive dashboard (also saved as outputs/dashboard.html)

In [55]:
print(ask("Which villages cannot reach a hospital by road, and how many buildings were hit?", RESULT["facts"], "en"))
print(ask("कुन बस्तीहरू अस्पताल पुग्न सक्दैनन्?", RESULT["facts"], "ne"))

The village 热索 cannot reach a hospital by road.  
Buildings hit: 30.  

These results are satellite‑derived estimates from an educational prototype.
बस्ती **热索** अस्पताल पुग्न असमर्थ छ।  
(परिणामहरू शैक्षिक प्रोटोटाइपबाट प्राप्त स्याटेलाइट‑आधारित अनुमान हुन्।)


In [56]:
import zipfile, glob, pandas as pd, geopandas as gpd
for z in glob.glob("/content/EMSR927_AOI0*.zip"):
    zipfile.ZipFile(z).extractall("/content/emsr927/" + z.split("/")[-1][:-4])

def load(pat):
    fs = sorted(glob.glob(f"/content/emsr927/**/*{pat}*.shp", recursive=True))
    print(pat, "->", [f.split("/")[-1] for f in fs])
    return pd.concat([gpd.read_file(f) for f in fs], ignore_index=True)

aoi, ev = load("areaOfInterestA"), load("observedEventA")     # AOI polygons + mapped event extent
ev.to_file("/content/emsr927_extent.geojson", driver="GeoJSON")

flood_in_aoi = gpd.clip(RESULT["flood"], aoi.to_crs(RESULT["flood"].crs))   # score only where the reference looked
print("reference extent km2: %.2f" % (ev.to_crs(32645).area.sum() / 1e6))
print("your mask inside the AOIs km2: %.3f" % (flood_in_aoi.to_crs(32645).area.sum() / 1e6))
print(evaluate_against_reference(flood_in_aoi, "/content/emsr927_extent.geojson", RESULT["grid"]))

areaOfInterestA -> ['EMSR927_AOI01_GRA_PRODUCT_areaOfInterestA_v1.shp', 'EMSR927_AOI02_GRA_PRODUCT_areaOfInterestA_v2.shp']
observedEventA -> ['EMSR927_AOI01_GRA_PRODUCT_observedEventA_v1.shp', 'EMSR927_AOI02_GRA_PRODUCT_observedEventA_v2.shp']
reference extent km2: 2.40
your mask inside the AOIs km2: 0.000
{'IoU': 0.0, 'precision': 0.0, 'recall': 0.0, 'F1': 0.0}


In [57]:
b, r, fz, st = damage_assessment(RESULT["osm"], RESULT["flood"])
pts = load("builtUpP").to_crs(b.crs)
dam = pts[pts.damage_gra.isin(["Destroyed", "Damaged"])]
b_in = gpd.clip(b, aoi.to_crs(b.crs))
print("OSM buildings inside AOIs:", len(b_in), "| flagged as hit:", int(b_in.hit.sum()))
print("reference destroyed/damaged buildings:", len(dam),
      "| possibly damaged:", int((pts.damage_gra == "Possibly damaged").sum()))
print("reference destroyed bridges: 6 | yours flagged:", st["bridges_hit"])

builtUpP -> ['EMSR927_AOI01_GRA_PRODUCT_builtUpP_v1.shp', 'EMSR927_AOI02_GRA_PRODUCT_builtUpP_v2.shp']
OSM buildings inside AOIs: 170 | flagged as hit: 0
reference destroyed/damaged buildings: 760 | possibly damaged: 104
reference destroyed bridges: 6 | yours flagged: 1


In [58]:
# Download everything (dashboard.html, situation_report.html, facts.json, flood.geojson)
import shutil
shutil.make_archive("floodwatch_outputs", "zip", CONFIG["out_dir"])
try:
    from google.colab import files; files.download("floodwatch_outputs.zip")
except Exception: pass

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>